In [ ]:
pip install pyampute

In [ ]:
import os
import pandas as pd
import numpy as np
import pyampute
from pyampute import MultivariateAmputation

def load_complete_datasets(iteration):
    """
    Load the complete train and test datasets for a specific iteration.
    """
    # define paths
    iter_dir = f"data/complete/iteration_{iteration}"
    train_path = f"{iter_dir}/train_complete.csv"
    test_path = f"{iter_dir}/test_complete.csv"
    
    if not os.path.exists(train_path) or not os.path.exists(test_path):
        raise FileNotFoundError(f"Complete datasets for iteration {iteration} not found in {iter_dir}")
    
    # load datasets
    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)
    
    print(f"Loaded complete datasets from iteration {iteration}")
    print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")
    
    return train_data, test_data
    
def apply_amputation(df, mcar_features, mar_features, mnar_features, prop, seed=57):
    """
    Apply missingness to selected features.
    """
    # get feature names excluding target
    features = [col for col in df.columns if col != 'target']
    
    # define missingness patterns
    patterns = [
        {"incomplete_vars": mcar_features,
         "mechanism": "MCAR",
         "freq": 0.33
        },
        {"incomplete_vars": mar_features,
         "mechanism": "MAR",
         "freq": 0.33,
         "weights": {"continuous_7": 1.5,
                     "continuous_9": 1.3,
                     "continuous_11": 1.1,
                     "discrete_3": 1.0}
         },  
         {"incomplete_vars": mnar_features,
          "mechanism": "MNAR",
          "freq": 0.34,
          "weights": {"continuous_10": 1.2,
                      "continuous_12": 1.2,
                      "continuous_14": 1.3,
                      "discrete_4": 1.0}
          }
    ] 
    
    # create amputer and apply missingness
    amputer = MultivariateAmputation(prop=prop, patterns=patterns, seed=seed)
    features_with_missing = amputer.fit_transform(df[features])
    
    # create new dataframe with missing values
    df_with_missing = df.copy()
    df_with_missing[features] = features_with_missing
    
    # calculate missingness statistics
    missing_count = df_with_missing.drop('target', axis=1).isnull().sum().sum()
    total_cells = df_with_missing.shape[0] * (df_with_missing.shape[1] - 1)
    missing_rate = (missing_count / total_cells) * 100
    
    # create missingness report
    missing_report = {
        'missing_count': missing_count,
        'total_cells': total_cells,
        'missing_rate': missing_rate
    }
    
    return df_with_missing, missing_report

def find_prop(df, mcar_features, mar_features, mnar_features, target_rate, seed):
    """
    Binary search to find prop value that produces missing rate closest to target_rate.
    """
    # initial search range
    low, high = 0.01, 1.0
    best_df, best_report, best_prop, best_diff = None, None, None, float('inf')
    
    for _ in range(25):  
        # choose midpoint
        mid = (low + high) / 2
        # apply missingness with current prop
        df_missing, report = apply_amputation(df, mcar_features, mar_features, mnar_features, mid, seed)
        # calculate how far from target rate
        diff = abs(report['missing_rate'] - target_rate)
        # track the best result seen so far
        if diff < best_diff:
            best_diff = diff
            best_df, best_report, best_prop = df_missing, report, mid
        
        # adjust search bounds
        if report['missing_rate'] < target_rate:
            low = mid
        else:
            high = mid
              
    return best_df, best_report, best_prop

def create_missing_datasets(train_data, test_data, iteration, random_seed):
    """
    Create datasets with low (10%) and high (25%) missing datasets with MCAR/MAR/MNAR.
    """
    print(f"Creating missing datasets for iteration {iteration} with seed {random_seed}")
    
    # create copies of data
    train_1 = train_data.copy()
    test_1 = test_data.copy()
    train_2 = train_data.copy()
    test_2 = test_data.copy()
    
    # define features for missing data mechanisms
    mcar_features = ['continuous_0', 'continuous_1', 'continuous_2', 'continuous_6', 'continuous_8', 'discrete_2', 'categorical_0_Yes', 'categorical_3_Yes']  
    mar_features = ['continuous_3', 'continuous_4', 'continuous_5', 'continuous_13', 'discrete_0', 'discrete_1', 'categorical_2_Yes', 'categorical_4_Level_B', 'categorical_4_Level_C']
    mnar_features = ['continuous_10', 'continuous_12', 'continuous_14', 'discrete_4']
    
    # apply low missing rate (10%)
    print("Applying low missing rate (target: ~10%):")
    train_low, train_low_report, train_low_prop = find_prop(train_1, mcar_features, mar_features, mnar_features, 10.0, random_seed)
    test_low, test_low_report, test_low_prop = find_prop(test_1, mcar_features, mar_features, mnar_features, 10.0, random_seed)
    print(f"Train low missing rate: {train_low_report['missing_rate']:.2f}%")
    print(f"Test low missing rate: {test_low_report['missing_rate']:.2f}%")
    
    # apply high missing rate (25%)
    print("Applying high missing rate (target: ~25%):")
    train_high, train_high_report, train_high_prop = find_prop(train_2, mcar_features, mar_features, mnar_features, 25.0, random_seed)
    test_high, test_high_report, test_high_prop = find_prop(test_2, mcar_features, mar_features, mnar_features, 25.0, random_seed)
    print(f"Train high missing rate: {train_high_report['missing_rate']:.2f}%")
    print(f"Test high missing rate: {test_high_report['missing_rate']:.2f}%")
    
    return train_low, test_low, train_high, test_high

def save_missing_datasets(train_low, test_low, train_high, test_high, iteration):
    """
    Save datasets with missing values.
    """
    # create iteration directory
    iter_dir = f"data/amputed/iteration_{iteration}"
    os.makedirs(iter_dir, exist_ok=True)
    
    # save files
    train_low.to_csv(f"{iter_dir}/train_low.csv", index=False)
    test_low.to_csv(f"{iter_dir}/test_low.csv", index=False)
    train_high.to_csv(f"{iter_dir}/train_high.csv", index=False)
    test_high.to_csv(f"{iter_dir}/test_high.csv", index=False)
    
    print(f"Saved missing datasets to {iter_dir}/")

if __name__ == "__main__":
    for seed in range(1, 6):
        print(f"Applying missingness for iteration {seed}")

        try:
            # 1. load the complete datasets for this iteration
            train_data, test_data = load_complete_datasets(seed)
            # 2. apply missingness
            train_low, test_low, train_high, test_high = create_missing_datasets(train_data, test_data, iteration=seed, random_seed=seed)
            # 3. save new datasets
            save_missing_datasets(train_low, test_low, train_high, test_high, seed)
        
            print(f"Successfully created missing datasets for iteration {seed}\n")
    
        except FileNotFoundError as e:
            print(f"Error: {e}")
            print("Please generate complete datasets first using data_generation.py")

    print("Successfully generated missing datasets for all 5 iterations.")